# Elastic Net Regression

A comprehensive guide to Elastic Net regularization, combining L1 (Lasso) and L2 (Ridge) penalties for robust linear regression.

---

## Table of Contents
1. [Theory Section](#1.-Theory-Section)
2. [Implementation from Scratch](#2.-Implementation-from-Scratch)
3. [Training & Optimization](#3.-Training-&-Optimization)
4. [Diagnostics & Evaluation](#4.-Diagnostics-&-Evaluation)
5. [Visualizations](#5.-Visualizations)
6. [Use Cases & Guidelines](#6.-Use-Cases-&-Guidelines)
7. [Comparison with sklearn](#7.-Comparison-with-sklearn)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

---

## 1. Theory Section

### 1.1 What is Elastic Net?

Elastic Net is a regularized regression method that linearly combines the L1 (Lasso) and L2 (Ridge) penalties. It was introduced by Zou and Hastie (2005) to address limitations of both Lasso and Ridge regression.

### 1.2 Mathematical Formulation

The Elastic Net objective function is:

$$\min_{\beta} \frac{1}{2n} ||y - X\beta||_2^2 + \alpha \left( \rho ||\beta||_1 + \frac{(1-\rho)}{2} ||\beta||_2^2 \right)$$

Where:
- $\alpha$ (alpha): Overall regularization strength
- $\rho$ (l1_ratio): Mixing parameter between L1 and L2
  - $\rho = 1$: Pure Lasso (L1)
  - $\rho = 0$: Pure Ridge (L2)
  - $0 < \rho < 1$: Elastic Net

### 1.3 The L1 and L2 Penalties

| Property | L1 (Lasso) | L2 (Ridge) | Elastic Net |
|----------|------------|------------|-------------|
| Sparsity | Yes | No | Yes |
| Feature Selection | Yes | No | Yes |
| Handles Correlated Features | Poorly | Well | Well |
| Grouping Effect | No | Yes | Yes |
| Unique Solution | Not always | Always | Always |

### 1.4 When is Elastic Net Better?

**Elastic Net excels when:**
1. **Correlated Features**: Unlike Lasso, which arbitrarily selects one feature from a correlated group, Elastic Net tends to select all correlated features together (grouping effect)
2. **p >> n**: When the number of features exceeds the number of samples
3. **Need for Sparsity + Stability**: Combines Lasso's feature selection with Ridge's stability

### 1.5 Coordinate Descent Algorithm

Elastic Net is typically solved using coordinate descent, which optimizes one coefficient at a time while holding others fixed.

For each coefficient $\beta_j$, the update rule is:

$$\beta_j \leftarrow \frac{S(\rho_j, \alpha \rho)}{1 + \alpha(1-\rho)}$$

Where $S$ is the soft-thresholding operator:

$$S(z, \gamma) = \text{sign}(z) \max(|z| - \gamma, 0)$$

And $\rho_j$ is the partial residual for feature $j$.

---

## 2. Implementation from Scratch

### 2.1 Elastic Net Class with Coordinate Descent

In [ ]:
class ElasticNetFromScratch:
    """
    Elastic Net regression using coordinate descent optimization.
    
    Parameters
    ----------
    alpha : float, default=1.0
        Regularization strength. Must be positive.
    l1_ratio : float, default=0.5
        Mixing parameter between L1 and L2 regularization.
        l1_ratio=1 is Lasso, l1_ratio=0 is Ridge.
    max_iter : int, default=1000
        Maximum number of iterations for coordinate descent.
    tol : float, default=1e-4
        Tolerance for convergence.
    fit_intercept : bool, default=True
        Whether to fit an intercept term.
    warm_start : bool, default=False
        Whether to reuse previous solution as initialization.
    """
    
    def __init__(
        self,
        alpha: float = 1.0,
        l1_ratio: float = 0.5,
        max_iter: int = 1000,
        tol: float = 1e-4,
        fit_intercept: bool = True,
        warm_start: bool = False
    ):
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.max_iter = max_iter
        self.tol = tol
        self.fit_intercept = fit_intercept
        self.warm_start = warm_start
        
        # Model parameters (set during fit)
        self.coef_ = None
        self.intercept_ = 0.0
        self.n_iter_ = 0
        self.convergence_history_ = []
        
        # Standardization parameters
        self._X_mean = None
        self._X_std = None
        self._y_mean = None
    
    @staticmethod
    def _soft_threshold(z: float, gamma: float) -> float:
        """
        Soft-thresholding operator for L1 regularization.
        
        S(z, gamma) = sign(z) * max(|z| - gamma, 0)
        """
        if z > gamma:
            return z - gamma
        elif z < -gamma:
            return z + gamma
        else:
            return 0.0
    
    def _standardize(self, X: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Standardize features and center target."""
        self._X_mean = X.mean(axis=0)
        self._X_std = X.std(axis=0)
        # Avoid division by zero for constant features
        self._X_std[self._X_std == 0] = 1.0
        
        X_scaled = (X - self._X_mean) / self._X_std
        
        self._y_mean = y.mean()
        y_centered = y - self._y_mean
        
        return X_scaled, y_centered
    
    def _coordinate_descent(
        self,
        X: np.ndarray,
        y: np.ndarray,
        coef: np.ndarray
    ) -> np.ndarray:
        """
        Perform coordinate descent optimization.
        
        Updates one coefficient at a time while holding others fixed.
        """
        n_samples, n_features = X.shape
        
        # Precompute X^T X diagonal for efficiency
        X_col_norms_sq = (X ** 2).sum(axis=0)
        
        # Regularization components
        l1_penalty = self.alpha * self.l1_ratio
        l2_penalty = self.alpha * (1 - self.l1_ratio)
        
        self.convergence_history_ = []
        
        for iteration in range(self.max_iter):
            coef_old = coef.copy()
            
            # Update each coefficient
            for j in range(n_features):
                # Compute partial residual (excluding feature j)
                residual = y - X @ coef + X[:, j] * coef[j]
                
                # Compute update (correlation with residual)
                rho_j = X[:, j] @ residual / n_samples
                
                # Apply soft-thresholding and L2 scaling
                z_j = X_col_norms_sq[j] / n_samples + l2_penalty
                
                if z_j != 0:
                    coef[j] = self._soft_threshold(rho_j, l1_penalty) / z_j
                else:
                    coef[j] = 0.0
            
            # Check convergence
            coef_change = np.max(np.abs(coef - coef_old))
            self.convergence_history_.append(coef_change)
            
            if coef_change < self.tol:
                self.n_iter_ = iteration + 1
                break
        else:
            self.n_iter_ = self.max_iter
        
        return coef
    
    def fit(self, X: np.ndarray, y: np.ndarray) -> 'ElasticNetFromScratch':
        """
        Fit the Elastic Net model.
        
        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)
            Training data.
        y : np.ndarray of shape (n_samples,)
            Target values.
        
        Returns
        -------
        self : ElasticNetFromScratch
            Fitted model.
        """
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        
        n_samples, n_features = X.shape
        
        # Standardize data
        X_scaled, y_centered = self._standardize(X, y)
        
        # Initialize coefficients
        if self.warm_start and self.coef_ is not None:
            coef = self.coef_.copy()
        else:
            coef = np.zeros(n_features)
        
        # Run coordinate descent
        coef = self._coordinate_descent(X_scaled, y_centered, coef)
        
        # Transform coefficients back to original scale
        self.coef_ = coef / self._X_std
        
        # Compute intercept
        if self.fit_intercept:
            self.intercept_ = self._y_mean - self._X_mean @ self.coef_
        else:
            self.intercept_ = 0.0
        
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Predict using the fitted model.
        
        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)
            Samples to predict.
        
        Returns
        -------
        y_pred : np.ndarray of shape (n_samples,)
            Predicted values.
        """
        X = np.asarray(X, dtype=np.float64)
        return X @ self.coef_ + self.intercept_
    
    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """
        Compute R^2 score.
        
        Parameters
        ----------
        X : np.ndarray
            Test samples.
        y : np.ndarray
            True values.
        
        Returns
        -------
        score : float
            R^2 score.
        """
        y = np.asarray(y).ravel()
        y_pred = self.predict(X)
        
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        
        return 1 - ss_res / ss_tot if ss_tot != 0 else 0.0
    
    def get_objective_value(self, X: np.ndarray, y: np.ndarray) -> float:
        """
        Compute the Elastic Net objective function value.
        """
        y = np.asarray(y).ravel()
        y_pred = self.predict(X)
        n_samples = len(y)
        
        mse = np.sum((y - y_pred) ** 2) / (2 * n_samples)
        l1_term = self.alpha * self.l1_ratio * np.sum(np.abs(self.coef_))
        l2_term = self.alpha * (1 - self.l1_ratio) * np.sum(self.coef_ ** 2) / 2
        
        return mse + l1_term + l2_term

### 2.2 Helper Function for Generating Synthetic Data with Correlated Features

In [ ]:
def generate_correlated_data(
    n_samples: int = 200,
    n_features: int = 20,
    n_informative: int = 10,
    n_groups: int = 3,
    group_correlation: float = 0.8,
    noise_std: float = 1.0,
    random_state: Optional[int] = None
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate synthetic regression data with correlated feature groups.
    
    This type of data is ideal for demonstrating Elastic Net's advantages
    over pure Lasso or Ridge regression.
    
    Parameters
    ----------
    n_samples : int
        Number of samples.
    n_features : int
        Total number of features.
    n_informative : int
        Number of features that contribute to the target.
    n_groups : int
        Number of correlated feature groups.
    group_correlation : float
        Correlation within groups (0 to 1).
    noise_std : float
        Standard deviation of noise in target.
    random_state : int, optional
        Random seed.
    
    Returns
    -------
    X : np.ndarray of shape (n_samples, n_features)
        Feature matrix.
    y : np.ndarray of shape (n_samples,)
        Target values.
    true_coef : np.ndarray of shape (n_features,)
        True coefficients used to generate data.
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    # Generate base features
    X = np.random.randn(n_samples, n_features)
    
    # Create correlated groups
    features_per_group = n_features // n_groups
    
    for g in range(n_groups):
        start_idx = g * features_per_group
        end_idx = start_idx + features_per_group
        
        # Create a base feature for this group
        base = np.random.randn(n_samples)
        
        # Mix base with independent noise to create correlated features
        for j in range(start_idx, end_idx):
            X[:, j] = group_correlation * base + np.sqrt(1 - group_correlation**2) * X[:, j]
    
    # Generate true coefficients (sparse)
    true_coef = np.zeros(n_features)
    informative_idx = np.random.choice(n_features, n_informative, replace=False)
    true_coef[informative_idx] = np.random.randn(n_informative) * 2
    
    # Generate target
    y = X @ true_coef + noise_std * np.random.randn(n_samples)
    
    return X, y, true_coef


def train_test_split(
    X: np.ndarray,
    y: np.ndarray,
    test_size: float = 0.2,
    random_state: Optional[int] = None
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Simple train-test split."""
    if random_state is not None:
        np.random.seed(random_state)
    
    n_samples = len(y)
    n_test = int(n_samples * test_size)
    
    indices = np.random.permutation(n_samples)
    test_idx = indices[:n_test]
    train_idx = indices[n_test:]
    
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

### 2.3 Quick Test of Implementation

In [ ]:
# Generate test data
X_test, y_test, true_coef_test = generate_correlated_data(
    n_samples=200, n_features=20, n_informative=8,
    n_groups=4, group_correlation=0.7, random_state=42
)

# Fit model
en = ElasticNetFromScratch(alpha=0.1, l1_ratio=0.5)
en.fit(X_test, y_test)

print("Elastic Net from Scratch - Quick Test")
print("=" * 40)
print(f"Converged in {en.n_iter_} iterations")
print(f"R^2 Score: {en.score(X_test, y_test):.4f}")
print(f"Non-zero coefficients: {np.sum(np.abs(en.coef_) > 1e-6)}")
print(f"Intercept: {en.intercept_:.4f}")

---

## 3. Training & Optimization

### 3.1 Generate Synthetic Data with Correlated Features

In [ ]:
# Generate data with correlated feature groups
X, y, true_coef = generate_correlated_data(
    n_samples=500,
    n_features=30,
    n_informative=12,
    n_groups=5,
    group_correlation=0.85,
    noise_std=1.5,
    random_state=42
)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Dataset Summary")
print("=" * 40)
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")
print(f"True non-zero coefficients: {np.sum(np.abs(true_coef) > 0)}")

# Visualize feature correlations
fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = np.corrcoef(X_train.T)
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=14)
ax.set_xlabel('Feature Index')
ax.set_ylabel('Feature Index')
plt.colorbar(im, ax=ax, label='Correlation')
plt.tight_layout()
plt.show()

### 3.2 Grid Search for Optimal Hyperparameters

In [ ]:
def cross_validate(
    X: np.ndarray,
    y: np.ndarray,
    alpha: float,
    l1_ratio: float,
    n_folds: int = 5,
    random_state: int = 42
) -> Tuple[float, float]:
    """
    Perform k-fold cross-validation for Elastic Net.
    
    Returns mean MSE and standard deviation.
    """
    np.random.seed(random_state)
    n_samples = len(y)
    indices = np.random.permutation(n_samples)
    fold_size = n_samples // n_folds
    
    mse_scores = []
    
    for fold in range(n_folds):
        # Create fold indices
        val_start = fold * fold_size
        val_end = val_start + fold_size
        
        val_idx = indices[val_start:val_end]
        train_idx = np.concatenate([indices[:val_start], indices[val_end:]])
        
        # Train model
        model = ElasticNetFromScratch(alpha=alpha, l1_ratio=l1_ratio)
        model.fit(X[train_idx], y[train_idx])
        
        # Evaluate
        y_pred = model.predict(X[val_idx])
        mse = np.mean((y[val_idx] - y_pred) ** 2)
        mse_scores.append(mse)
    
    return np.mean(mse_scores), np.std(mse_scores)


# Define parameter grid
alphas = np.logspace(-3, 1, 15)
l1_ratios = np.linspace(0.1, 0.9, 9)

print("Running Grid Search with Cross-Validation...")
print(f"Alpha values: {len(alphas)}, L1 ratio values: {len(l1_ratios)}")

# Store results
results = np.zeros((len(alphas), len(l1_ratios)))
results_std = np.zeros((len(alphas), len(l1_ratios)))

for i, alpha in enumerate(alphas):
    for j, l1_ratio in enumerate(l1_ratios):
        mean_mse, std_mse = cross_validate(X_train, y_train, alpha, l1_ratio)
        results[i, j] = mean_mse
        results_std[i, j] = std_mse

# Find best parameters
best_idx = np.unravel_index(np.argmin(results), results.shape)
best_alpha = alphas[best_idx[0]]
best_l1_ratio = l1_ratios[best_idx[1]]

print(f"\nBest Parameters:")
print(f"  Alpha: {best_alpha:.4f}")
print(f"  L1 Ratio: {best_l1_ratio:.2f}")
print(f"  CV MSE: {results[best_idx]:.4f} (+/- {results_std[best_idx]:.4f})")

### 3.3 Train Final Model with Best Parameters

In [ ]:
# Train final model
final_model = ElasticNetFromScratch(alpha=best_alpha, l1_ratio=best_l1_ratio)
final_model.fit(X_train, y_train)

# Evaluate on test set
y_pred_train = final_model.predict(X_train)
y_pred_test = final_model.predict(X_test)

train_mse = np.mean((y_train - y_pred_train) ** 2)
test_mse = np.mean((y_test - y_pred_test) ** 2)
train_r2 = final_model.score(X_train, y_train)
test_r2 = final_model.score(X_test, y_test)

print("Final Model Performance")
print("=" * 40)
print(f"Training MSE: {train_mse:.4f}")
print(f"Test MSE: {test_mse:.4f}")
print(f"Training R^2: {train_r2:.4f}")
print(f"Test R^2: {test_r2:.4f}")
print(f"Iterations to converge: {final_model.n_iter_}")
print(f"Non-zero coefficients: {np.sum(np.abs(final_model.coef_) > 1e-6)} / {len(final_model.coef_)}")

---

## 4. Diagnostics & Evaluation

### 4.1 MSE vs Alpha and L1 Ratio

In [ ]:
def compute_mse_grid(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    alphas: np.ndarray,
    l1_ratios: np.ndarray
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute train and test MSE for all combinations of alpha and l1_ratio.
    """
    train_mse = np.zeros((len(alphas), len(l1_ratios)))
    test_mse = np.zeros((len(alphas), len(l1_ratios)))
    
    for i, alpha in enumerate(alphas):
        for j, l1_ratio in enumerate(l1_ratios):
            model = ElasticNetFromScratch(alpha=alpha, l1_ratio=l1_ratio)
            model.fit(X_train, y_train)
            
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)
            
            train_mse[i, j] = np.mean((y_train - y_pred_train) ** 2)
            test_mse[i, j] = np.mean((y_test - y_pred_test) ** 2)
    
    return train_mse, test_mse


# Compute MSE grid
alphas_diag = np.logspace(-3, 1, 20)
l1_ratios_diag = np.linspace(0.0, 1.0, 11)

train_mse_grid, test_mse_grid = compute_mse_grid(
    X_train, y_train, X_test, y_test, alphas_diag, l1_ratios_diag
)

# Plot MSE vs Alpha for different l1_ratios
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Select specific l1_ratios to plot
plot_l1_ratios = [0.0, 0.3, 0.5, 0.7, 1.0]
colors = plt.cm.viridis(np.linspace(0, 1, len(plot_l1_ratios)))

for idx, l1_ratio in enumerate(plot_l1_ratios):
    j = np.argmin(np.abs(l1_ratios_diag - l1_ratio))
    label = f'l1_ratio={l1_ratio:.1f}'
    if l1_ratio == 0.0:
        label += ' (Ridge)'
    elif l1_ratio == 1.0:
        label += ' (Lasso)'
    
    axes[0].semilogx(alphas_diag, train_mse_grid[:, j], '-o', 
                     color=colors[idx], label=label, markersize=4)
    axes[1].semilogx(alphas_diag, test_mse_grid[:, j], '-o',
                     color=colors[idx], label=label, markersize=4)

axes[0].set_xlabel('Alpha (log scale)')
axes[0].set_ylabel('MSE')
axes[0].set_title('Training MSE vs Alpha')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Alpha (log scale)')
axes[1].set_ylabel('MSE')
axes[1].set_title('Test MSE vs Alpha')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 Coefficient Stability Analysis

In [ ]:
def compute_coefficient_stability(
    X: np.ndarray,
    y: np.ndarray,
    alpha: float,
    l1_ratio: float,
    n_bootstrap: int = 50,
    random_state: int = 42
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute coefficient stability using bootstrap resampling.
    
    Returns mean and std of coefficients across bootstrap samples.
    """
    np.random.seed(random_state)
    n_samples, n_features = X.shape
    
    coef_samples = np.zeros((n_bootstrap, n_features))
    
    for b in range(n_bootstrap):
        # Bootstrap sample
        idx = np.random.choice(n_samples, n_samples, replace=True)
        
        model = ElasticNetFromScratch(alpha=alpha, l1_ratio=l1_ratio)
        model.fit(X[idx], y[idx])
        coef_samples[b] = model.coef_
    
    return coef_samples.mean(axis=0), coef_samples.std(axis=0)


# Compare stability: Lasso vs Ridge vs Elastic Net
print("Computing coefficient stability (bootstrap)...")

stability_results = {}
configs = [
    ('Ridge (l1=0.0)', 0.1, 0.0),
    ('Elastic Net (l1=0.5)', 0.1, 0.5),
    ('Lasso (l1=1.0)', 0.1, 1.0)
]

for name, alpha, l1_ratio in configs:
    mean_coef, std_coef = compute_coefficient_stability(
        X_train, y_train, alpha, l1_ratio, n_bootstrap=30
    )
    stability_results[name] = {'mean': mean_coef, 'std': std_coef}
    print(f"  {name}: Mean coef std = {std_coef.mean():.4f}")

# Visualize stability
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, data) in zip(axes, stability_results.items()):
    x = np.arange(len(data['mean']))
    ax.bar(x, data['mean'], yerr=data['std'], capsize=2, alpha=0.7, color='steelblue')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Feature Index')
    ax.set_ylabel('Coefficient Value')
    ax.set_title(f'{name}\nCoefficient Stability')
    ax.set_xticks(x[::5])

plt.tight_layout()
plt.show()

# Summary statistics
print("\nCoefficient Stability Summary")
print("=" * 50)
for name, data in stability_results.items():
    avg_std = data['std'].mean()
    max_std = data['std'].max()
    n_stable = np.sum(data['std'] < 0.5)
    print(f"{name}:")
    print(f"  Avg coef std: {avg_std:.4f}")
    print(f"  Max coef std: {max_std:.4f}")
    print(f"  Stable coefficients (std < 0.5): {n_stable}/{len(data['std'])}")

### 4.3 Convergence Analysis

In [ ]:
# Analyze convergence for different configurations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Different alphas with fixed l1_ratio
alphas_conv = [0.001, 0.01, 0.1, 1.0]
colors = plt.cm.plasma(np.linspace(0.2, 0.8, len(alphas_conv)))

for alpha, color in zip(alphas_conv, colors):
    model = ElasticNetFromScratch(alpha=alpha, l1_ratio=0.5, max_iter=200)
    model.fit(X_train, y_train)
    axes[0].semilogy(model.convergence_history_, color=color, label=f'alpha={alpha}')

axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Max Coefficient Change (log scale)')
axes[0].set_title('Convergence vs Alpha (l1_ratio=0.5)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Different l1_ratios with fixed alpha
l1_ratios_conv = [0.0, 0.25, 0.5, 0.75, 1.0]
colors = plt.cm.viridis(np.linspace(0, 1, len(l1_ratios_conv)))

for l1_ratio, color in zip(l1_ratios_conv, colors):
    model = ElasticNetFromScratch(alpha=0.1, l1_ratio=l1_ratio, max_iter=200)
    model.fit(X_train, y_train)
    label = f'l1_ratio={l1_ratio}'
    axes[1].semilogy(model.convergence_history_, color=color, label=label)

axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Max Coefficient Change (log scale)')
axes[1].set_title('Convergence vs L1 Ratio (alpha=0.1)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 5. Visualizations

### 5.1 2D Heatmap of Performance vs (Alpha, L1 Ratio)

In [ ]:
# Create detailed heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Test MSE Heatmap
im1 = axes[0].imshow(
    test_mse_grid.T,
    aspect='auto',
    origin='lower',
    cmap='RdYlGn_r',
    extent=[np.log10(alphas_diag[0]), np.log10(alphas_diag[-1]), 
            l1_ratios_diag[0], l1_ratios_diag[-1]]
)

# Mark best point
best_idx_test = np.unravel_index(np.argmin(test_mse_grid), test_mse_grid.shape)
axes[0].scatter(
    np.log10(alphas_diag[best_idx_test[0]]),
    l1_ratios_diag[best_idx_test[1]],
    marker='*', s=300, c='blue', edgecolors='white', linewidths=2,
    label=f'Best: MSE={test_mse_grid[best_idx_test]:.3f}'
)

axes[0].set_xlabel('log10(Alpha)')
axes[0].set_ylabel('L1 Ratio')
axes[0].set_title('Test MSE Heatmap\n(lower is better)')
axes[0].legend(loc='upper right')
plt.colorbar(im1, ax=axes[0], label='MSE')

# Sparsity Heatmap (number of non-zero coefficients)
sparsity_grid = np.zeros((len(alphas_diag), len(l1_ratios_diag)))

for i, alpha in enumerate(alphas_diag):
    for j, l1_ratio in enumerate(l1_ratios_diag):
        model = ElasticNetFromScratch(alpha=alpha, l1_ratio=l1_ratio)
        model.fit(X_train, y_train)
        sparsity_grid[i, j] = np.sum(np.abs(model.coef_) > 1e-6)

im2 = axes[1].imshow(
    sparsity_grid.T,
    aspect='auto',
    origin='lower',
    cmap='viridis',
    extent=[np.log10(alphas_diag[0]), np.log10(alphas_diag[-1]),
            l1_ratios_diag[0], l1_ratios_diag[-1]]
)

axes[1].set_xlabel('log10(Alpha)')
axes[1].set_ylabel('L1 Ratio')
axes[1].set_title('Number of Non-Zero Coefficients\n(Sparsity)')
plt.colorbar(im2, ax=axes[1], label='Non-zero coefficients')

# Add annotations for special regions
axes[1].axhline(y=0.0, color='white', linestyle='--', alpha=0.5, label='Ridge (l1=0)')
axes[1].axhline(y=1.0, color='white', linestyle='-.', alpha=0.5, label='Lasso (l1=1)')

plt.tight_layout()
plt.show()

### 5.2 Coefficient Paths

In [ ]:
def compute_coefficient_path(
    X: np.ndarray,
    y: np.ndarray,
    alphas: np.ndarray,
    l1_ratio: float
) -> np.ndarray:
    """
    Compute coefficients for a range of alpha values.
    Uses warm starts for efficiency.
    """
    n_features = X.shape[1]
    coef_path = np.zeros((len(alphas), n_features))
    
    # Sort alphas in decreasing order for warm start efficiency
    alpha_order = np.argsort(alphas)[::-1]
    
    prev_model = None
    for idx in alpha_order:
        alpha = alphas[idx]
        model = ElasticNetFromScratch(alpha=alpha, l1_ratio=l1_ratio, warm_start=True)
        
        if prev_model is not None:
            model.coef_ = prev_model.coef_.copy()
        
        model.fit(X, y)
        coef_path[idx] = model.coef_
        prev_model = model
    
    return coef_path


# Compute coefficient paths for different l1_ratios
alphas_path = np.logspace(-4, 1, 50)
l1_ratios_path = [0.1, 0.5, 0.9]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, l1_ratio in zip(axes, l1_ratios_path):
    coef_path = compute_coefficient_path(X_train, y_train, alphas_path, l1_ratio)
    
    # Plot each coefficient
    for j in range(coef_path.shape[1]):
        ax.semilogx(alphas_path, coef_path[:, j], alpha=0.7, linewidth=1.5)
    
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Alpha (log scale)')
    ax.set_ylabel('Coefficient Value')
    
    if l1_ratio == 0.1:
        title = f'L1 Ratio = {l1_ratio} (Ridge-like)'
    elif l1_ratio == 0.9:
        title = f'L1 Ratio = {l1_ratio} (Lasso-like)'
    else:
        title = f'L1 Ratio = {l1_ratio} (Balanced)'
    ax.set_title(f'Coefficient Path\n{title}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.3 Comparison of True vs Estimated Coefficients

In [ ]:
# Train models with different configurations
models = {
    'Ridge (l1=0.0)': ElasticNetFromScratch(alpha=best_alpha, l1_ratio=0.0),
    'Elastic Net (l1=0.5)': ElasticNetFromScratch(alpha=best_alpha, l1_ratio=0.5),
    'Lasso (l1=1.0)': ElasticNetFromScratch(alpha=best_alpha, l1_ratio=1.0),
    f'Best EN (l1={best_l1_ratio:.1f})': ElasticNetFromScratch(alpha=best_alpha, l1_ratio=best_l1_ratio)
}

for model in models.values():
    model.fit(X_train, y_train)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

x = np.arange(len(true_coef))
width = 0.35

for ax, (name, model) in zip(axes, models.items()):
    ax.bar(x - width/2, true_coef, width, label='True', alpha=0.7, color='green')
    ax.bar(x + width/2, model.coef_, width, label='Estimated', alpha=0.7, color='blue')
    
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Feature Index')
    ax.set_ylabel('Coefficient Value')
    ax.set_title(f'{name}\nTest R^2: {model.score(X_test, y_test):.3f}')
    ax.legend(loc='upper right')
    ax.set_xticks(x[::5])

plt.tight_layout()
plt.show()

### 5.4 Regularization Path 3D Surface

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Create 3D surface plot
fig = plt.figure(figsize=(14, 6))

# Prepare meshgrid
log_alphas = np.log10(alphas_diag)
A, L = np.meshgrid(log_alphas, l1_ratios_diag)

# Test MSE Surface
ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(A, L, test_mse_grid.T, cmap='RdYlGn_r', alpha=0.8, edgecolors='gray', linewidth=0.2)
ax1.set_xlabel('log10(Alpha)')
ax1.set_ylabel('L1 Ratio')
ax1.set_zlabel('Test MSE')
ax1.set_title('Test MSE Surface')
ax1.view_init(elev=25, azim=45)

# Sparsity Surface
ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(A, L, sparsity_grid.T, cmap='viridis', alpha=0.8, edgecolors='gray', linewidth=0.2)
ax2.set_xlabel('log10(Alpha)')
ax2.set_ylabel('L1 Ratio')
ax2.set_zlabel('Non-zero Coefficients')
ax2.set_title('Sparsity Surface')
ax2.view_init(elev=25, azim=45)

plt.tight_layout()
plt.show()

---

## 6. Use Cases & Guidelines

### 6.1 When to Use Elastic Net

**Ideal Scenarios:**

1. **Highly Correlated Features**
   - When features are grouped and correlated, Elastic Net tends to select or reject groups together
   - This "grouping effect" is often desirable in real-world applications

2. **High-Dimensional Data (p >> n)**
   - When you have more features than samples
   - Lasso can select at most n features; Elastic Net has no such limitation

3. **Need Both Sparsity and Stability**
   - Want feature selection (from L1) but also numerical stability (from L2)
   - Useful in production systems where model stability matters

4. **Multicollinearity**
   - When predictors are highly correlated
   - Ridge's L2 penalty helps stabilize coefficient estimates

In [ ]:
# Demonstration: Grouping Effect
print("Demonstrating the Grouping Effect")
print("=" * 50)

# Create data with two perfectly correlated features
np.random.seed(42)
n = 100
X_group = np.random.randn(n, 5)
X_group[:, 1] = X_group[:, 0] + 0.01 * np.random.randn(n)  # Feature 1 ~ Feature 0
X_group[:, 3] = X_group[:, 2] + 0.01 * np.random.randn(n)  # Feature 3 ~ Feature 2

# Target depends on all features
true_coef_group = np.array([1.0, 1.0, 2.0, 2.0, 0.5])
y_group = X_group @ true_coef_group + 0.1 * np.random.randn(n)

# Fit different models
results_group = {}
for name, l1_ratio in [('Ridge', 0.0), ('Elastic Net', 0.5), ('Lasso', 1.0)]:
    model = ElasticNetFromScratch(alpha=0.1, l1_ratio=l1_ratio)
    model.fit(X_group, y_group)
    results_group[name] = model.coef_

# Compare coefficients
print("\nCoefficient Comparison (Features 0 & 1 are correlated, 2 & 3 are correlated):\n")
print(f"{'Feature':<10} {'True':<10} {'Ridge':<10} {'Elastic Net':<12} {'Lasso':<10}")
print("-" * 52)
for i in range(5):
    print(f"{i:<10} {true_coef_group[i]:<10.3f} {results_group['Ridge'][i]:<10.3f} "
          f"{results_group['Elastic Net'][i]:<12.3f} {results_group['Lasso'][i]:<10.3f}")

print("\nObservation: Lasso tends to select only one feature from correlated pairs,")
print("while Elastic Net distributes weight across correlated features (grouping effect).")

### 6.2 When NOT to Use Elastic Net

**Avoid Elastic Net when:**

1. **Features are Independent**
   - If features are not correlated, Lasso alone may suffice
   - Elastic Net's L2 component adds unnecessary complexity

2. **Pure Ridge is Sufficient**
   - If you don't need sparsity (all features are important)
   - Ridge has a closed-form solution and is faster

3. **Interpretability is Critical**
   - Lasso's strict sparsity (exactly zero coefficients) is easier to interpret
   - Elastic Net may keep more non-zero (but small) coefficients

4. **Computational Constraints**
   - Two hyperparameters require more tuning (cross-validation)
   - Ridge and Lasso are simpler with one hyperparameter each

In [ ]:
# Demonstration: When Lasso is Sufficient
print("Demonstration: Independent Features (Lasso may suffice)")
print("=" * 50)

# Create data with independent features
np.random.seed(42)
X_indep = np.random.randn(200, 20)
true_coef_indep = np.zeros(20)
true_coef_indep[:5] = np.array([3.0, -2.0, 1.5, -1.0, 0.5])
y_indep = X_indep @ true_coef_indep + 0.5 * np.random.randn(200)

# Split
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_indep, y_indep, test_size=0.2)

# Compare models
print("\nTest R^2 Scores:")
for name, l1_ratio in [('Ridge', 0.0), ('Elastic Net (0.5)', 0.5), ('Lasso', 1.0)]:
    model = ElasticNetFromScratch(alpha=0.05, l1_ratio=l1_ratio)
    model.fit(X_train_i, y_train_i)
    r2 = model.score(X_test_i, y_test_i)
    n_nonzero = np.sum(np.abs(model.coef_) > 1e-6)
    print(f"  {name:<20}: R^2 = {r2:.4f}, Non-zero = {n_nonzero}")

print("\nWith independent features, all methods perform similarly.")
print("Lasso provides the cleanest sparsity pattern.")

### 6.3 Parameter Selection Strategies

**Strategy 1: Sequential Search**
1. Fix l1_ratio = 0.5 (balanced)
2. Search for optimal alpha using CV
3. Fix alpha, search for optimal l1_ratio
4. Repeat if necessary

**Strategy 2: Grid Search**
- Search over 2D grid of (alpha, l1_ratio)
- More computationally expensive but thorough

**Strategy 3: Heuristics**
- Start with l1_ratio = 0.5 for balance
- Increase l1_ratio if you need more sparsity
- Decrease l1_ratio if coefficients are unstable

In [ ]:
def sequential_parameter_search(
    X: np.ndarray,
    y: np.ndarray,
    alphas: np.ndarray,
    l1_ratios: np.ndarray,
    n_iterations: int = 3
) -> Tuple[float, float]:
    """
    Sequential parameter search: alternate between optimizing alpha and l1_ratio.
    """
    # Initialize with default values
    best_alpha = 0.1
    best_l1_ratio = 0.5
    
    for iteration in range(n_iterations):
        # Optimize alpha with fixed l1_ratio
        best_mse = float('inf')
        for alpha in alphas:
            mse, _ = cross_validate(X, y, alpha, best_l1_ratio, n_folds=3)
            if mse < best_mse:
                best_mse = mse
                best_alpha = alpha
        
        # Optimize l1_ratio with fixed alpha
        best_mse = float('inf')
        for l1_ratio in l1_ratios:
            mse, _ = cross_validate(X, y, best_alpha, l1_ratio, n_folds=3)
            if mse < best_mse:
                best_mse = mse
                best_l1_ratio = l1_ratio
    
    return best_alpha, best_l1_ratio


# Compare search strategies
print("Parameter Selection Strategy Comparison")
print("=" * 50)

# Sequential search
import time
start = time.time()
seq_alpha, seq_l1 = sequential_parameter_search(
    X_train, y_train,
    np.logspace(-3, 1, 10),
    np.linspace(0.1, 0.9, 5)
)
seq_time = time.time() - start

# Grid search (already done above)
print(f"\nSequential Search: alpha={seq_alpha:.4f}, l1_ratio={seq_l1:.2f}")
print(f"  Time: {seq_time:.2f}s")
print(f"\nGrid Search: alpha={best_alpha:.4f}, l1_ratio={best_l1_ratio:.2f}")

# Evaluate both
model_seq = ElasticNetFromScratch(alpha=seq_alpha, l1_ratio=seq_l1)
model_seq.fit(X_train, y_train)

model_grid = ElasticNetFromScratch(alpha=best_alpha, l1_ratio=best_l1_ratio)
model_grid.fit(X_train, y_train)

print(f"\nTest R^2 - Sequential: {model_seq.score(X_test, y_test):.4f}")
print(f"Test R^2 - Grid: {model_grid.score(X_test, y_test):.4f}")

---

## 7. Comparison with sklearn

### 7.1 Basic Comparison

In [ ]:
from sklearn.linear_model import ElasticNet as SklearnElasticNet
from sklearn.preprocessing import StandardScaler

# Compare implementations
print("Comparison: Custom Implementation vs sklearn")
print("=" * 60)

# Test parameters
test_alpha = 0.1
test_l1_ratio = 0.5

# Our implementation
our_model = ElasticNetFromScratch(alpha=test_alpha, l1_ratio=test_l1_ratio)
our_model.fit(X_train, y_train)

# sklearn implementation
sklearn_model = SklearnElasticNet(alpha=test_alpha, l1_ratio=test_l1_ratio, max_iter=1000)
sklearn_model.fit(X_train, y_train)

print(f"\nParameters: alpha={test_alpha}, l1_ratio={test_l1_ratio}")
print(f"\n{'Metric':<25} {'Custom':<15} {'sklearn':<15}")
print("-" * 55)

# Compare metrics
metrics = [
    ('Train R^2', our_model.score(X_train, y_train), sklearn_model.score(X_train, y_train)),
    ('Test R^2', our_model.score(X_test, y_test), sklearn_model.score(X_test, y_test)),
    ('Train MSE', np.mean((y_train - our_model.predict(X_train))**2),
                  np.mean((y_train - sklearn_model.predict(X_train))**2)),
    ('Test MSE', np.mean((y_test - our_model.predict(X_test))**2),
                 np.mean((y_test - sklearn_model.predict(X_test))**2)),
    ('Intercept', our_model.intercept_, sklearn_model.intercept_),
    ('Non-zero coefs', np.sum(np.abs(our_model.coef_) > 1e-6),
                       np.sum(np.abs(sklearn_model.coef_) > 1e-6)),
    ('Iterations', our_model.n_iter_, sklearn_model.n_iter_)
]

for name, our_val, sklearn_val in metrics:
    if isinstance(our_val, float):
        print(f"{name:<25} {our_val:<15.4f} {sklearn_val:<15.4f}")
    else:
        print(f"{name:<25} {our_val:<15} {sklearn_val:<15}")

### 7.2 Coefficient Comparison

In [ ]:
# Visual comparison of coefficients
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Coefficient values comparison
x = np.arange(len(our_model.coef_))
width = 0.35

axes[0].bar(x - width/2, our_model.coef_, width, label='Custom', alpha=0.7)
axes[0].bar(x + width/2, sklearn_model.coef_, width, label='sklearn', alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('Coefficient Value')
axes[0].set_title('Coefficient Comparison')
axes[0].legend()
axes[0].set_xticks(x[::5])

# Scatter plot of coefficients
axes[1].scatter(sklearn_model.coef_, our_model.coef_, alpha=0.7, s=50)
min_val = min(sklearn_model.coef_.min(), our_model.coef_.min())
max_val = max(sklearn_model.coef_.max(), our_model.coef_.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect match')
axes[1].set_xlabel('sklearn Coefficients')
axes[1].set_ylabel('Custom Coefficients')
axes[1].set_title('Coefficient Correlation')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Coefficient difference
coef_diff = our_model.coef_ - sklearn_model.coef_
axes[2].bar(x, coef_diff, alpha=0.7, color='orange')
axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[2].set_xlabel('Feature Index')
axes[2].set_ylabel('Coefficient Difference')
axes[2].set_title(f'Custom - sklearn\n(Max diff: {np.max(np.abs(coef_diff)):.6f})')
axes[2].set_xticks(x[::5])

plt.tight_layout()
plt.show()

# Numerical comparison
print(f"\nCoefficient Comparison Statistics:")
print(f"  Mean Absolute Difference: {np.mean(np.abs(coef_diff)):.6f}")
print(f"  Max Absolute Difference: {np.max(np.abs(coef_diff)):.6f}")
print(f"  Correlation: {np.corrcoef(our_model.coef_, sklearn_model.coef_)[0, 1]:.6f}")

### 7.3 Performance Comparison Across Parameter Space

In [ ]:
# Compare across different parameters
alphas_comp = [0.01, 0.1, 1.0]
l1_ratios_comp = [0.2, 0.5, 0.8]

print("Performance Comparison Across Parameters")
print("=" * 80)
print(f"{'Alpha':<10} {'L1 Ratio':<10} {'Custom R^2':<15} {'sklearn R^2':<15} {'Diff':<10}")
print("-" * 60)

for alpha in alphas_comp:
    for l1_ratio in l1_ratios_comp:
        # Custom
        custom = ElasticNetFromScratch(alpha=alpha, l1_ratio=l1_ratio)
        custom.fit(X_train, y_train)
        custom_r2 = custom.score(X_test, y_test)
        
        # sklearn
        sklearn = SklearnElasticNet(alpha=alpha, l1_ratio=l1_ratio)
        sklearn.fit(X_train, y_train)
        sklearn_r2 = sklearn.score(X_test, y_test)
        
        diff = abs(custom_r2 - sklearn_r2)
        print(f"{alpha:<10} {l1_ratio:<10} {custom_r2:<15.4f} {sklearn_r2:<15.4f} {diff:<10.6f}")

### 7.4 Timing Comparison

In [ ]:
import time

# Generate larger dataset for timing
X_large, y_large, _ = generate_correlated_data(
    n_samples=2000, n_features=100, n_informative=30,
    n_groups=10, group_correlation=0.8, random_state=42
)

print("Timing Comparison (2000 samples, 100 features)")
print("=" * 50)

# Custom implementation
times_custom = []
for _ in range(5):
    start = time.time()
    model = ElasticNetFromScratch(alpha=0.1, l1_ratio=0.5)
    model.fit(X_large, y_large)
    times_custom.append(time.time() - start)

# sklearn implementation
times_sklearn = []
for _ in range(5):
    start = time.time()
    model = SklearnElasticNet(alpha=0.1, l1_ratio=0.5)
    model.fit(X_large, y_large)
    times_sklearn.append(time.time() - start)

print(f"\nCustom Implementation:")
print(f"  Mean: {np.mean(times_custom)*1000:.2f} ms")
print(f"  Std:  {np.std(times_custom)*1000:.2f} ms")

print(f"\nsklearn Implementation:")
print(f"  Mean: {np.mean(times_sklearn)*1000:.2f} ms")
print(f"  Std:  {np.std(times_sklearn)*1000:.2f} ms")

print(f"\nSpeed ratio (sklearn/custom): {np.mean(times_custom)/np.mean(times_sklearn):.2f}x")
print("\nNote: sklearn is optimized with Cython for better performance.")

### 7.5 sklearn Cross-Validation with ElasticNetCV

In [ ]:
from sklearn.linear_model import ElasticNetCV

# Use sklearn's built-in cross-validation
print("sklearn ElasticNetCV (Automatic Parameter Selection)")
print("=" * 50)

# Define parameter grid
sklearn_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
    alphas=np.logspace(-4, 1, 20),
    cv=5,
    max_iter=1000
)

sklearn_cv.fit(X_train, y_train)

print(f"\nBest Parameters (sklearn CV):")
print(f"  Alpha: {sklearn_cv.alpha_:.4f}")
print(f"  L1 Ratio: {sklearn_cv.l1_ratio_:.2f}")
print(f"\nTest R^2: {sklearn_cv.score(X_test, y_test):.4f}")
print(f"Non-zero coefficients: {np.sum(np.abs(sklearn_cv.coef_) > 1e-6)}")

# Compare with our grid search results
print(f"\nComparison with Custom Grid Search:")
print(f"  Custom - Alpha: {best_alpha:.4f}, L1 Ratio: {best_l1_ratio:.2f}")
print(f"  sklearn - Alpha: {sklearn_cv.alpha_:.4f}, L1 Ratio: {sklearn_cv.l1_ratio_:.2f}")

---

## Summary

### Key Takeaways

1. **Elastic Net combines L1 and L2 regularization**, providing both feature selection and coefficient stability.

2. **The l1_ratio parameter** controls the balance:
   - l1_ratio = 1.0: Pure Lasso (maximum sparsity)
   - l1_ratio = 0.0: Pure Ridge (maximum stability)
   - 0 < l1_ratio < 1: Elastic Net (balanced)

3. **Grouping effect**: Elastic Net tends to select correlated features together, unlike Lasso which arbitrarily picks one.

4. **Coordinate descent** is the standard optimization algorithm, updating one coefficient at a time.

5. **Parameter tuning** requires searching over both alpha and l1_ratio (use cross-validation).

6. **Use Elastic Net when**:
   - Features are correlated
   - You need both sparsity and stability
   - p >> n (high-dimensional data)

7. **Consider alternatives when**:
   - Features are independent (Lasso suffices)
   - No sparsity needed (Ridge is simpler)
   - Strict interpretability required (Lasso's exact zeros)

### References

- Zou, H., & Hastie, T. (2005). Regularization and variable selection via the elastic net. *Journal of the Royal Statistical Society: Series B*, 67(2), 301-320.
- Friedman, J., Hastie, T., & Tibshirani, R. (2010). Regularization paths for generalized linear models via coordinate descent. *Journal of Statistical Software*, 33(1), 1-22.